In [1]:
import os
import logging
import argparse
import pandas as pd
from dotenv import load_dotenv
from neo4j_handler import Neo4jHandler

# Import Splink components using the confirmed syntax
import splink.comparison_library as cl
from splink import DuckDBAPI, Linker, SettingsCreator, block_on, splink_datasets

# Configure logging.
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Load environment variables from the .env file.
load_dotenv()
NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USER = os.getenv("NEO4J_USER")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")
if not (NEO4J_URI and NEO4J_USER and NEO4J_PASSWORD):
    raise ValueError("Please set NEO4J_URI, NEO4J_USER, and NEO4J_PASSWORD in your .env file")

handler = Neo4jHandler(NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD)

INFO:neo4j_handler:Connected to Neo4j at bolt://localhost:7687 as user neo4j


In [2]:
def fetch_identities(handler):
    """
    Fetch all Identity nodes from the Neo4j database and return them as a list of dictionaries.
    Expected columns: id, full_name, email_address, zip_code, phone_number.
    """
    query = """
    MATCH (i:Identity)
    RETURN i.id AS id,
           i.id as unique_id,
           i.full_name AS full_name,
           i.email_address AS email_address,
           i.zip_code AS zip_code,
           i.phone_number AS phone_number,
           i.wcc_component as   wcc_component
    """
    result = handler.execute_read(query)
    # Consume the result within the transaction scope.
    records = [record.data() for record in result]
    return records


identities = fetch_identities(handler)

In [9]:
identities_df = pd.DataFrame(identities)

identities_df.head()



,id,unique_id,full_name,email_address,zip_code,phone_number,wcc_component
0,74755953-4f79-447c-8316-1015652ed505,74755953-4f79-447c-8316-1015652ed505,Myrtle Flores,myrtle.flores@example.cmo,14548,2639301607,0
1,61312ef2-484b-4344-8962-2ffaf80989dd,61312ef2-484b-4344-8962-2ffaf80989dd,Terrance Morgan,terrance.morgan@example.com,77611,3347003876,1
2,86a12fd1-3f54-4fa3-943e-c04061ea97e8,86a12fd1-3f54-4fa3-943e-c04061ea97e8,Andrea Hall,andrea.hall@exapmle.com,69692,5957849360,2
3,c6d313cd-bbe0-4605-992c-3286edef9ff0,c6d313cd-bbe0-4605-992c-3286edef9ff0,Monica Day,monica.day@example.co,17325,9477307354,3
4,5d89769a-cb30-4c97-8816-ba1c69694654,5d89769a-cb30-4c97-8816-ba1c69694654,Bessie Peterson,bessie.peterson@example.cgm,49857,2892830278,4


In [16]:
"""
Extract Identity nodes from Neo4j, run entity resolution using Splink 4.0.7 with DuckDB,
and return the resulting clusters as a Pandas DataFrame.
"""
logger.info("Fetching Identity nodes from Neo4j...")
identities = fetch_identities(handler)
if not identities:
    logger.info("No Identity nodes found.")


# Convert the list of identity records into a Pandas DataFrame.
df = pd.DataFrame(identities)

# Add a source_dataset column to the DataFrame.
df["source_dataset"] = "dataset_1"  # Assign a default value for all records.

logger.info(f"Fetched {len(df)} identity records.")

# Create Splink settings using the new syntax.
# Here we use our existing columns:
# - NameComparison on full_name
# - EmailComparison on email_address
# - ExactMatch on zip_code and phone_number (with term frequency adjustments for zip_code)
settings = SettingsCreator(
    link_type="dedupe_only",
    comparisons=[
        cl.NameComparison("full_name"),
        cl.EmailComparison("email_address"),
        cl.LevenshteinAtThresholds("zip_code").configure(term_frequency_adjustments=True),
        cl.LevenshteinAtThresholds("phone_number"),
    ],
    blocking_rules_to_generate_predictions=[
        block_on("wcc_component"),  
    ]
)

logger.info("Initializing DuckDB backend for Splink...")
db_api = DuckDBAPI()  # Create a DuckDB backend instance


logger.info("Initializing Splink Linker...")
# Initialize the generic Linker with the DataFrame, settings, and DuckDB backend.
linker = Linker(df, settings, db_api=db_api)



INFO:__main__:Fetching Identity nodes from Neo4j...
INFO:__main__:Fetched 3790 identity records.
INFO:__main__:Initializing DuckDB backend for Splink...
INFO:__main__:Initializing Splink Linker...


In [17]:
# -------------------------
# Training steps:
# Estimate the probability that two random records match.
linker.training.estimate_probability_two_random_records_match(
    [block_on("wcc_component")],
    recall=0.7,
)

INFO:splink.internals.linker_components.training:Probability two random records match is estimated to be  0.0127.
This means that amongst all possible pairwise record comparisons, one in 78.96 are expected to match.  With 7,180,155 total possible comparisons, we expect a total of around 90,937.14 matching pairs


In [18]:
# Estimate u probabilities using random sampling.
linker.training.estimate_u_using_random_sampling(max_pairs=1e8)


INFO:splink.internals.estimate_u:----- Estimating u probabilities using random sampling -----
INFO:splink.internals.estimate_u:
Estimated u probabilities using random sampling
INFO:splink.internals.settings:
Your model is not yet fully trained. Missing estimates for:
    - full_name (no m values are trained).
    - email_address (no m values are trained).
    - zip_code (no m values are trained).
    - phone_number (no m values are trained).


In [27]:

# Estimate m probabilities (true match likelihoods) using expectation maximisation.
for property in ["full_name", "email_address", "zip_code", "phone_number"]:
    linker.training.estimate_parameters_using_expectation_maximisation(block_on(property))
# linker.training.estimate_parameters_using_expectation_maximisation(block_on("wcc_component"))


INFO:splink.internals.em_training_session:
----- Starting EM training session -----



INFO:splink.internals.em_training_session:Estimating the m probabilities of the model by blocking on:
l."full_name" = r."full_name"

Parameter estimates will be made for the following comparison(s):
    - email_address
    - zip_code
    - phone_number

Parameter estimates cannot be made for the following comparison(s) since they are used in the blocking rules: 
INFO:splink.internals.expectation_maximisation:
Level All other comparisons on comparison email_address not observed in dataset, unable to train m value

INFO:splink.internals.expectation_maximisation:Iteration 1: Largest change in params was 0.191 in probability_two_random_records_match
INFO:splink.internals.expectation_maximisation:Iteration 2: Largest change in params was 5.48e-08 in probability_two_random_records_match
INFO:splink.internals.expectation_maximisation:
EM converged after 2 iterations
INFO:splink.internals.em_training_session:m probability not trained for email_address - All other comparisons (comparison vector

In [28]:

# -------------------------
# Prediction and Clustering:
# Predict pairwise match scores.
pairwise_predictions = linker.inference.predict(threshold_match_weight=-5)
# Cluster the predictions at a given threshold (e.g. 0.95)
clusters = linker.clustering.cluster_pairwise_predictions_at_threshold(
    pairwise_predictions, 0.95
)
# Convert the clusters to a Pandas DataFrame.
df_clusters = clusters.as_pandas_dataframe()


INFO:splink.internals.linker_components.inference:Blocking time: 0.06 seconds
INFO:splink.internals.linker_components.inference:Predict time: 0.48 seconds
INFO:splink.internals.connected_components:Completed iteration 1, num representatives needing updating: 0


In [29]:
df_clusters.head()

,cluster_id,id,unique_id,full_name,email_address,zip_code,phone_number,wcc_component,source_dataset
0,250f5ef6-3732-48df-82e8-0803178420ff,4f2c3e56-3a22-4fc7-8e8b-db4ead94acb2,4f2c3e56-3a22-4fc7-8e8b-db4ead94acb2,Marshall Holland,marshall.holland@example.com,60206,8559358011,5,dataset_1
1,250f5ef6-3732-48df-82e8-0803178420ff,4eefe9a7-e491-4292-bec6-d258fda11066,4eefe9a7-e491-4292-bec6-d258fda11066,Marshall Holland,marshall.holland@example.com,60206,8559358011,5,dataset_1
2,250f5ef6-3732-48df-82e8-0803178420ff,8a8d7363-c9dc-43a6-b9cc-8ffa97662389,8a8d7363-c9dc-43a6-b9cc-8ffa97662389,Marshall Holland,marshall.holland@example.com,60206,8559358011,5,dataset_1
3,250f5ef6-3732-48df-82e8-0803178420ff,aa853979-bfdb-46da-ab07-4bfbebac0bb5,aa853979-bfdb-46da-ab07-4bfbebac0bb5,Marshall Holland,marshall.holland@example.com,60206,8559358011,5,dataset_1
4,250f5ef6-3732-48df-82e8-0803178420ff,d5085a82-697f-4d3a-8cb5-249ba25f1458,d5085a82-697f-4d3a-8cb5-249ba25f1458,Marshall Holland,marshall.holland@example.com,60206,8559358011,5,dataset_1


In [30]:
# write the clusters to neo4j
rows = df_clusters.to_dict("records")

# Cypher query to update nodes: it unwinds each row and sets the cluster_id property.
update_query = """
UNWIND $rows as row
MATCH (n {id: row.id})
SET n.cluster_id = row.cluster_id
RETURN count(n) AS updated_count
"""

# Use your Neo4jHandler's execute_write method to run the update.
handler.execute_write(update_query, parameters={"rows": rows})